# Documentation Crazyflie Results

This notebook creates the numbers and plots used for the result documentation of the Crazyflie. Make sure all evaluated results are stored as pickl files in `evaluation_data/rotorpy/`

In [ ]:
from pathlib import Path

from evaluation.core import EvaluationResult
from evaluation.visualizer import EvaluationVisualizer

EVAL_DATA_PATH = Path("../../evaluation_data/rotorpy")
NO_BIAS_EVAL_DATA_PATH = Path("../../evaluation_data/rotorpy/no_bias")
REFERENCE_SCENARIO = "crazyflie_evaluation"


def _load_dir(path: Path) -> list[EvaluationResult]:
    files = sorted(path.glob("*.pkl"))
    if not files:
        print(f"  {path.name}/: (empty — run 99_crazyflie_multi_evaluation first)")
        return []
    results = [EvaluationResult.load(p) for p in files]
    print(f"  {path.name}/: {[p.stem for p in files]}")
    return results


def _ref_eval(evals: list[EvaluationResult]) -> EvaluationResult | None:
    return next(
        (e for e in evals if e.metadata["simulation_metadata"]["name"] == REFERENCE_SCENARIO),
        evals[0] if evals else None,
    )


print("\nVariant evaluations:")
no_tof_evaluations: list[EvaluationResult] = _load_dir(EVAL_DATA_PATH / "no_tof")
tof_evaluations: list[EvaluationResult] = _load_dir(EVAL_DATA_PATH / "tof")
tof_sw_evaluations: list[EvaluationResult] = _load_dir(EVAL_DATA_PATH / "tof_sw")
tof_ema_evaluations: list[EvaluationResult] = _load_dir(EVAL_DATA_PATH / "tof_ema")

no_tof_ref_evaluation: EvaluationResult = _ref_eval(no_tof_evaluations)
tof_ref_evaluation: EvaluationResult = _ref_eval(tof_evaluations)
tof_sw_ref_evaluation: EvaluationResult = _ref_eval(tof_sw_evaluations)
tof_ema_ref_evaluation: EvaluationResult = _ref_eval(tof_ema_evaluations)

print("\nNo-bias Variant evaluations:")
no_bias_tof_evaluations: list[EvaluationResult] = _load_dir(NO_BIAS_EVAL_DATA_PATH / "tof")
no_bias_tof_sw_evaluations: list[EvaluationResult] = _load_dir(NO_BIAS_EVAL_DATA_PATH / "tof_sw")
no_bias_tof_ema_evaluations: list[EvaluationResult] = _load_dir(NO_BIAS_EVAL_DATA_PATH / "tof_ema")

no_bias_tof_ref_evaluation: EvaluationResult = _ref_eval(no_bias_tof_evaluations)
no_bias_tof_sw_ref_evaluation: EvaluationResult = _ref_eval(no_bias_tof_sw_evaluations)
no_bias_tof_ema_ref_evaluation: EvaluationResult = _ref_eval(no_bias_tof_ema_evaluations)

## Flight Trajectories

In [ ]:
PLOT_VARIANT = "tof_sw"  # "no_tof" | "tof" | "tof_sw" | "tof_ema"
SHOW_ESTIMATE = True

_variant_map = {
    "no_tof": no_tof_evaluations,
    "tof": tof_evaluations,
    "tof_sw": tof_sw_evaluations,
    "tof_ema": tof_ema_evaluations,
}
_plot_results = _variant_map[PLOT_VARIANT]
_plot_labels = [e.metadata["simulation_metadata"]["name"] for e in _plot_results]

# fig = EvaluationVisualizer.plot_multiple_flight_trajectories(
#     _plot_results,
#     labels=_plot_labels,
#     show_estimate=SHOW_ESTIMATE,
#     title=f"Simulated Crazyflie Trajectories",
# )
# fig.update_layout(
#       font=dict(size=16),
#       width=1200,
#       height=800,
#   )
# fig.show()

## Accuracy

In [ ]:
import numpy as np
import pandas as pd

METRICS = ["position_rmse", "velocity_rmse", "attitude_rmse"]

# Use the tof variant as the primary accuracy reference
evaluations = tof_evaluations

scenarios = [ev.metadata["simulation_metadata"]["name"] for ev in evaluations]
print(f"Averaging over {len(evaluations)} scenarios: {scenarios}\n")

collected = {m: {"x": [], "y": [], "z": [], "3D": []} for m in METRICS}
for ev in evaluations:
    for m in METRICS:
        r = ev.metrics[m]
        collected[m]["3D"].append(r.value)
        for ax in ("x", "y", "z"):
            collected[m][ax].append(r.per_axis[ax])

rows = []
for m in METRICS:
    unit = evaluations[0].metrics[m].unit
    row = {"Metric": m, "Unit": unit}
    for col in ("x", "y", "z", "3D"):
        vals = np.array(collected[m][col])
        row[col] = f"{vals.mean():.4f} ± {vals.std():.4f}"
    rows.append(row)

df = pd.DataFrame(rows).set_index("Metric")
df

In [ ]:
ax = EvaluationVisualizer.plot_altitude_error_comparison(
    [no_tof_ref_evaluation, tof_ref_evaluation],
    labels=["no ToF", "ToF"],
)
# ax.get_figure().savefig("crazyflie_altitude_error.png", dpi=300, bbox_inches="tight")

In [ ]:
import numpy as np
import pandas as pd

_alt_variants = [
    ("no ToF", no_tof_evaluations),
    ("ToF", tof_evaluations),
]

_rows = []
for variant_name, evals in _alt_variants:
    z_vals = np.array([e.metrics["position_rmse"].per_axis["z"] for e in evals])
    _rows.append({
        "Variant": variant_name,
        "Mean [m]": f"{z_vals.mean():.4f}",
        "Std [m]": f"{z_vals.std():.4f}",
        "Min [m]": f"{z_vals.min():.4f}",
        "Max [m]": f"{z_vals.max():.4f}",
        "n scenarios": len(evals),
    })

pd.DataFrame(_rows).set_index("Variant")

In [ ]:
visualizer = EvaluationVisualizer(tof_ref_evaluation)
ax = visualizer.plot_trajectory()
# ax.figure.savefig("crazyflie_gt_vs_estimate_trajectory.png", dpi=300, bbox_inches="tight")

In [ ]:
fig = visualizer.plot_state_errors()
# fig.savefig("crazyflie_state_error.png", dpi=600, bbox_inches="tight", pad_inches=0.3)

## Consistency

In [ ]:
visualizer = EvaluationVisualizer(tof_ref_evaluation)
ax = visualizer.plot_nees_groups_over_time()
# ax.figure.savefig("crazyflie_grouped_nees.png", dpi=300, bbox_inches="tight")

## Manual Optimization vs. Adaptive Measurement Noise

In [ ]:
import numpy as np
import pandas as pd

METRICS = ["position_rmse", "velocity_rmse", "attitude_rmse"]
VARIANT_EVALS = {
    # "no_tof": no_tof_evaluations,
    "tof": tof_evaluations,
    "tof_sw": tof_sw_evaluations,
    "tof_ema": tof_ema_evaluations,
}

rows = []
for m in METRICS:
    row = {"Metric": m}
    for variant_name, evals in VARIANT_EVALS.items():
        vals = np.array([e.metrics[m].value for e in evals])
        row[variant_name] = f"{vals.mean():.4f} ± {vals.std():.4f}"
    row["Unit"] = evals[0].metrics[m].unit
    rows.append(row)

df = pd.DataFrame(rows).set_index("Metric")
df

In [ ]:
fig = EvaluationVisualizer.plot_state_error_norms_comparison(
    [tof_ref_evaluation, tof_sw_ref_evaluation, tof_ema_ref_evaluation],
    labels=["Static", "SW", "EMA"],
)
# fig.savefig("crazyflie_adaptive_position_error.png", dpi=600, bbox_inches="tight", pad_inches=0.3)


In [ ]:
visualizer = EvaluationVisualizer(tof_sw_ref_evaluation)
visualizer.plot_measurement_variances()
print("plots")

In [ ]:
fig = EvaluationVisualizer.plot_state_error_norms_comparison(
    [tof_ref_evaluation, tof_sw_ref_evaluation, tof_ema_ref_evaluation, no_bias_tof_ref_evaluation, no_bias_tof_sw_ref_evaluation,
     no_bias_tof_ema_ref_evaluation],
    labels=["Static", "SW", "EMA", "Static (bias compensated)", "SW (bias compensated)", "EMA (bias compensated)"],
)
fig.savefig("crazyflie_adaptive_position_error_no_bias.png", dpi=600, bbox_inches="tight", pad_inches=0.3)

In [ ]:
visualizer = EvaluationVisualizer(no_bias_tof_ema_ref_evaluation)
visualizer.plot_measurement_variances()
print("plots")